In [1]:
function range(start: number, stop: number): number[] {
    return Array.from({ length: stop - start + 1 }, (_, index) => start + index);
}

# The Knight's Tour

This notebook computes a solution to the [knight's tour](https://en.wikipedia.org/wiki/Knight%27s_tour) using the constraint solver `Z3`.  

In [2]:
import { init, Bool, BitVec } from 'z3-solver';
const { Context } = await init();
const ctx = Context("main");

Given an integer from the set $\{0, 1, \cdots, 63\}$, the function `row(i)` computes the name of the variable that specifies the *row* of the knight after its $i^{\textrm{th}}$ move.

In [3]:
function row(i: number): string {
    return `R${i}`;
}

Given an integer from the set $\{0, 1, \cdots, 63\}$, the function `col(i)` computes the name of the variable that specifies the *column* of the knight after its $i^{\textrm{th}}$ move.

In [4]:
function col(i: number): string {
    return `C${i}`;
}

The function `isKnightMove(row, col, rowX, colX)` takes four arguments:
* `row` is a `Z3` variable that specifies the row of the position of the knight before the move.
* `col` is a `Z3` variable that specifies the column of the position of the knight before the move.
* `rowX` is a `Z3` variable that specifies the row of the position of the knight after the move.
* `colX` is a `Z3` variable that specifies the column of the position of the knight after the move.

It returns a formula that specifies that the specified position represents a legal move for a knight.

In order to form the *conjunction* of two formulas we use the method `ctx.And`, 
while the *disjunction* is build with the function `ctx.Or`.  
Note that these functions can be called with any number of arguments.

The figure below shows the moves of a knight:  The knight on `e4` can jump to all red squares.
<img src="knight-moves.png" width="50%">

The function `toBitVec(x)` turns the number `x` into a *bit vector* of length 4.

In [5]:
const toBitVec = (x: number): BitVec => ctx.BitVec.val(x, 4);

The function `toBitVar(x)` turns a string into a variable that has a type of bit vector of length 4.

In [6]:
const toBitVar = (x: string): BitVec => ctx.BitVec.const(x, 4);

In [7]:
const S = [1, 2, -1, -2];
S.flatMap(x => S.filter(y => Math.abs(x) != Math.abs(y)).map(y => [x, y]))

[
  [ 1, 2 ],  [ 1, -2 ],
  [ 2, 1 ],  [ 2, -1 ],
  [ -1, 2 ], [ -1, -2 ],
  [ -2, 1 ], [ -2, -1 ]
]


In [8]:
function isKnightMove(row: BitVec, col: BitVec, rowX: BitVec, colX: BitVec): Bool {
    const S = [1, 2, -1, -2];
    const deltaSet = S.flatMap(x => S.filter(y => Math.abs(x) != Math.abs(y)).map(y => [x, y]));
    const formulas = deltaSet.map(([delta_r, delta_c]) => 
        ctx.And(rowX.eq(row.add(toBitVec(delta_r))),
                colX.eq(col.add(toBitVec(delta_c))) ));
    return ctx.Or(...formulas);
}

The function `allDifferent` takes two arguments:
* `Rows` is a list of `Z3` variables. The variable `Rows[i]` specifies the row of the position of the knight after the $i^{\textrm{th}}$ move.
* `Cols` is a list of `Z3` variables. The variable `Cols[i]` specifies the column of the position of the knight after the $i^{\textrm{th}}$ move.

The function returns a set of formulas stating that for $i \not= j$ the positions after the $i^{\textrm{th}}$ move
differs from the position after the $j^{\textrm{th}}$ move.

In [9]:
function allDifferent(Rows: BitVec[], Cols: BitVec[]): Bool[] {
    return range(0, 62).flatMap(i => 
        range(i + 1, 63).map(j => 
            ctx.Or(Rows[i].neq(Rows[j]), Cols[i].neq(Cols[j]))
        )
    );
}

The function `allConstraints` takes two arguments:
* `Rows` is a list of `Z3` variables. The variable `Rows[i]` specifies the row of the position of the knight after the $i^{\textrm{th}}$ move.
* `Cols` is a list of `Z3` variables. The variable `Cols[i]` specifies the column of the position of the knight after the $i^{\textrm{th}}$ move.

`allConstraints` returns a set containing all constraints of the problem.

In [10]:
function allConstraints(Rows: BitVec[], Cols: BitVec[]): Bool[] {
    const zero  = toBitVec(0);
    const eight = toBitVec(8);
    return [...allDifferent(Rows, Cols), 
            Rows[0].eq(zero), Cols[0].eq(zero),
            ...range(0, 62).map(i => isKnightMove(Rows[i], Cols[i], Rows[i+1], Cols[i+1])),
            ...range(0, 63).flatMap(i => [Rows[i].ult(eight), Cols[i].ult(eight)])
           ];
}

In [11]:
function parseVal(expr: BitVec): number {
    return parseInt(expr.toString().substring(2), 16);
}

The function `solve()` computes a solution of the knight's problem and returns this solution.

In [12]:
async function solve(): Promise<[number, number][]> {
    const Rows: BitVec[] = Array.from({ length: 64 }, (_, i) => toBitVar(row(i)));
    const Cols: BitVec[] = Array.from({ length: 64 }, (_, i) => toBitVar(col(i)));
    const constraints = allConstraints(Rows, Cols);
    const solver = new ctx.Solver();
    solver.add(...constraints);
    const check = await solver.check();      
    if (check == 'sat') {
        const model = solver.model();
        const solution: [number, number][] = Array.from({ length: 64 }, (_, i) => {
            const r = parseVal(model.eval(Rows[i]));
            const c = parseVal(model.eval(Cols[i]));
            return [r, c];
        });
        return solution;
    } else if (check == 'unsat') {
        console.log('The problem is not solvable.');
    } else {
        console.log('Z3 cannot determine whether the problem is solvable.');
    }
}

Unfortunately, the execution time of the following cell varies greatly between
different runs.  Sometimes the cell runs in less one minute and 28 seconds, sometimes 
it might take 30 minutes.

In [13]:
console.time("Solver Duration");
const Solution = await solve();
console.timeEnd("Solver Duration");
Solution

Solver Duration: 1:07.114 (m:ss.mmm)
[
  [ 0, 0 ], [ 1, 2 ], [ 2, 4 ], [ 3, 6 ], [ 5, 5 ],
  [ 6, 7 ], [ 4, 6 ], [ 2, 7 ], [ 0, 6 ], [ 1, 4 ],
  [ 0, 2 ], [ 2, 3 ], [ 0, 4 ], [ 1, 6 ], [ 3, 7 ],
  [ 4, 5 ], [ 6, 4 ], [ 5, 2 ], [ 6, 0 ], [ 7, 2 ],
  [ 5, 1 ], [ 7, 0 ], [ 6, 2 ], [ 4, 1 ], [ 2, 0 ],
  [ 0, 1 ], [ 1, 3 ], [ 0, 5 ], [ 1, 7 ], [ 2, 5 ],
  [ 3, 3 ], [ 2, 1 ], [ 4, 2 ], [ 3, 0 ], [ 1, 1 ],
  [ 0, 3 ], [ 1, 5 ], [ 0, 7 ], [ 2, 6 ], [ 3, 4 ],
  [ 5, 3 ], [ 7, 4 ], [ 6, 6 ], [ 4, 7 ], [ 3, 5 ],
  [ 4, 3 ], [ 2, 2 ], [ 1, 0 ], [ 3, 1 ], [ 5, 0 ],
  [ 7, 1 ], [ 6, 3 ], [ 4, 4 ], [ 3, 2 ], [ 4, 0 ],
  [ 6, 1 ], [ 7, 3 ], [ 5, 4 ], [ 7, 5 ], [ 5, 6 ],
  [ 7, 7 ], [ 6, 5 ], [ 5, 7 ], [ 7, 6 ]
]


The function `createBoard(Solution)` returns a matrix `Board` of size $8\times 8$.
The following holds:
$$ \texttt{Board}[\texttt{R}i][\texttt{C}i] = i $$
Therefore, if `Board[r][c] == i`, then at the beginning of the $i^{\textrm{th}}$ move the knight is located in row `r` and column `c`. 

In [14]:
function createBoard(solution: [number, number][]) {
    const board: number[][] = Array.from({ length: 8 }, () => Array(8).fill(0));
    solution.forEach(([r, c], i) => { board[r][c] = i; });    
    return board;
}

In [15]:
createBoard(Solution)

[
  [
     0, 25, 10, 35,
    12, 27,  8, 37
  ],
  [
    47, 34,  1, 26,
     9, 36, 13, 28
  ],
  [
    24, 31, 46, 11,
     2, 29, 38,  7
  ],
  [
    33, 48, 53, 30,
    39, 44,  3, 14
  ],
  [
    54, 23, 32, 45,
    52, 15,  6, 43
  ],
  [
    49, 20, 17, 40,
    57,  4, 59, 62
  ],
  [
    18, 55, 22, 51,
    16, 61, 42,  5
  ],
  [
    21, 50, 19, 56,
    41, 58, 63, 60
  ]
]


The function `printBoard` prints the given `Board`.

In [16]:
function printBoard(board: number[][]) {
    const n = board.length;
    if (n === 0) return;
    const width = board
        .flat()
        .reduce((max, element) => Math.max(max, element.toString().length), 0);
    const createLine = (left: string, mid: string, right: string, fill: string) => {
        let line = left;
        for (let i = 0; i < n - 1; i++) {
            line += fill.repeat(width + 2) + mid;
        }
        line += fill.repeat(width + 2) + right;
        return line;
    };
    const topLine = createLine('╔', '╦', '╗', '═');
    const midLine = createLine('╠', '╬', '╣', '═');
    const botLine = createLine('╚', '╩', '╝', '═');
    console.log(topLine);
    board.forEach((row, i) => {
        let line = '\u2551';
        for (const element of row) {
            const s = element.toString().padStart(width, ' ');
            line += ` ${s} ║`;
        }
        console.log(line);
        if (i < n - 1) {
            console.log(midLine);
        }
    });
    console.log(botLine);
}

In [17]:
printBoard(createBoard(Solution))

╔════╦════╦════╦════╦════╦════╦════╦════╗
║  0 ║ 25 ║ 10 ║ 35 ║ 12 ║ 27 ║  8 ║ 37 ║
╠════╬════╬════╬════╬════╬════╬════╬════╣
║ 47 ║ 34 ║  1 ║ 26 ║  9 ║ 36 ║ 13 ║ 28 ║
╠════╬════╬════╬════╬════╬════╬════╬════╣
║ 24 ║ 31 ║ 46 ║ 11 ║  2 ║ 29 ║ 38 ║  7 ║
╠════╬════╬════╬════╬════╬════╬════╬════╣
║ 33 ║ 48 ║ 53 ║ 30 ║ 39 ║ 44 ║  3 ║ 14 ║
╠════╬════╬════╬════╬════╬════╬════╬════╣
║ 54 ║ 23 ║ 32 ║ 45 ║ 52 ║ 15 ║  6 ║ 43 ║
╠════╬════╬════╬════╬════╬════╬════╬════╣
║ 49 ║ 20 ║ 17 ║ 40 ║ 57 ║  4 ║ 59 ║ 62 ║
╠════╬════╬════╬════╬════╬════╬════╬════╣
║ 18 ║ 55 ║ 22 ║ 51 ║ 16 ║ 61 ║ 42 ║  5 ║
╠════╬════╬════╬════╬════╬════╬════╬════╣
║ 21 ║ 50 ║ 19 ║ 56 ║ 41 ║ 58 ║ 63 ║ 60 ║
╚════╩════╩════╩════╩════╩════╩════╩════╝


# Visualization

The function `showSolution` displays the given solution on a chessboard.
The solution `Board` is represented as a list of lists.  We have `Board[row][col] == k` if the $k^\textrm{th}$ move leads the knight to the position `(row, col)`.

In [18]:
import * as tslab from "tslab";

function showSolution(board: number[][], width: string = "50%"): void {
    const n = board.length;
    const cellSize = 50;
    const totalSize = n * cellSize;
    const path: [number, number][] = new Array(n * n);
    for (let r = 0; r < n; r++) {
        for (let c = 0; c < n; c++) {
            const k = board[r][c];
            if (k >= 0 && k < n * n) {
                path[k] = [r, c];
            }
        }
    }
    let svg = `<svg width="${width}" viewBox="0 0 ${totalSize} ${totalSize}" xmlns="http://www.w3.org/2000/svg" style="font-family: sans-serif;">`;
    for (let r = 0; r < n; r++) {
        for (let c = 0; c < n; c++) {
            const isBlack = (r + c) % 2 !== 0;
            const color = isBlack ? '#b58863' : '#f0d9b5';
            const x = c * cellSize;
            const y = r * cellSize;         
            svg += `<rect x="${x}" y="${y}" width="${cellSize}" height="${cellSize}" fill="${color}" />`;
            svg += `<text x="${x + cellSize/2}" y="${y + cellSize/2 + 5}" text-anchor="middle" font-size="14" fill="${isBlack ? '#f0d9b5' : '#b58863'}">${board[r][c]}</text>`;
        }
    }
    if (path[0]) {
        const [startR, startC] = path[0];
        svg += `<circle cx="${startC * cellSize + cellSize/2}" cy="${startR * cellSize + cellSize/2}" r="8" fill="darkblue" />`;
    }
    for (let k = 0; k < n * n - 1; k++) {
        const p1 = path[k];
        const p2 = path[k+1];
        if (p1 && p2) {
            const x1 = p1[1] * cellSize + cellSize/2;
            const y1 = p1[0] * cellSize + cellSize/2;
            const x2 = p2[1] * cellSize + cellSize/2;
            const y2 = p2[0] * cellSize + cellSize/2;
            svg += `<line x1="${x1}" y1="${y1}" x2="${x2}" y2="${y2}" stroke="darkblue" stroke-width="3" stroke-opacity="0.6" />`;
            svg += `<circle cx="${x2}" cy="${y2}" r="3" fill="darkblue" />`;
        }
    }
    svg += `</svg>`;
    tslab.display.html(svg);
}

In [19]:
showSolution(createBoard(Solution))

0 25 10 35 12 27 8 37 47 34 1 26 9 36 13 28 24 31 46 11 2 29 38 7 33 48 53 30 39 44 3 14 54 23 32 45 52 15 6 43 49 20 17 40 57 4 59 62 18 55 22 51 16 61 42 5 21 50 19 56 41 58 63 60